#WhisperX simple GUI

In [ ]:
# @title 의존성 설치
!pip install uv -q
!uv pip install whisperx yt-dlp[default,deno,curl_cffi] -q

In [ ]:
# @title 런타임 gpu 확인(선택사항)
!nvidia-smi
#@markdown - 해당 명령어가 작동하지 않는 등의 경우에는 다른 런타임 유형으로 바꿔야 함.
#@markdown - 무료 버전에서 사용 가능한 Nvidia gpu는 T4 GPU가 있음.


In [ ]:
# @title 링크 처리(오디오 다운로드, 제목 지정)
from pathlib import Path
import yt_dlp

contents_dir = Path("/content")

URLs = "" #@param {type:"string"}
#@markdown - 복수의 url은 띄어쓰기으로 구별.
#@markdown - 플래이리스트도 지원.
#@markdown - 이 칸을 비우고 이 셀을 실행시킨 후, 생성된 폴더에 수동으로 오디오 파일을 업로드해도 됨.

filename_format = '%(title)s.%(ext)s' #@param {type:"string"}
#@markdown - 예시: '%(title)s - %(uploader)s [%(upload_date>%Y.%m.%d)s].%(ext)s' 등. 사용 가능한 형식은 [깃허브](https://github.com/yt-dlp/yt-dlp#output-template) 참조.
concurrent_fragment_downloads = 16 #@param {type:"integer"}
quiet = True #@param {type:"boolean"}

audio_dir = contents_dir / "audio"
audio_dir.mkdir(parents=True, exist_ok=True)

ydl_opts = {
   'format': 'ba/worst',
   'outtmpl': {'default': filename_format},
   'paths': {'home': str(audio_dir)},
   'concurrent_fragment_downloads': concurrent_fragment_downloads
}
if quiet:
    ydl_opts['quiet'] = True

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    # for url in URLs.split("\n"):
        # info_dict = ydl.extract_info(url, download=True)
        # filepath = audio_dir / ydl.evaluate_outtmpl(filename_format, info_dict)
    if URLs:
        ydl.download(URLs.split(" "))
    # path 지정했으니까 따로 목록 리스트로 저장할 필요 없이 glob로 가져오기

# !yt-dlp -f "ba/worst" -N 16 --output '%(title)s.%(ext)s' {URL}
# worst는 유튜브가 아닌 플랫폼에서, 통합된 경우에 영상 품질과 용량이 가장 작은 파일로 선택하기 위함. 오디오가 그리 차이 안나는 경우가 많아서

# filename = !yt-dlp -f "ba/worst" --print '%(title)s.%(ext)s' {URL}
# filename = filename[0]
# inputpath = Path("/content") / f"{filename}"
# print(f"파일 경로: {inputpath}")

In [ ]:
# @title 다운로드된 파일 목록 확인
inputs = [str(f) for f in audio_dir.rglob("*") if f.is_file()]
print(f"확인된 파일 목록: ")
for idx, name in enumerate(inputs):
    print(f"{idx}: {name}")

In [ ]:
import math
# @title 전사 작업(기본 명령어)
model = "turbo" #@param ["tiny", "tiny.en", "base", "base.en", "small", "small.en", "distil-small.en", "medium", "medium.en", "distil-medium.en", "large-v1", "large-v2", "large-v3", "large", "distil-large-v2", "distil-large-v3", "large-v3-turbo", "turbo"]
#@markdown - 모델 별 성능과 메모리 사용량은 (ct2 변환되지 않은 경우의 비교이지만) [여기](https://github.com/openai/whisper#available-models-and-languages)를 참고.
_extra_model_hf = "" #@param {type:"string"}
if _extra_model_hf: model = _extra_model_hf
#@markdown - _extra_model_hf: [`RoachLin/kotoba-whisper-v2.2-faster`](https://huggingface.co/RoachLin/kotoba-whisper-v2.2-faster) 등 huggingface hub의 ct2 변환 모델 사용 가능
device = "cuda" #@param ["cpu","cuda"]
batch_size = 8 #@param {type:"integer"}
threads = 0 #@param {type:"integer"}
#@markdown - threads: `cpu`에서 작동.

compute_type = "default" #@param ["default","float16","float32","int8"]
fp16 = True #@param {type:"boolean"}
#@markdown - fp16: 추론을 fp16으로 할지 여부


_translate_to_en = False #@param {type:"boolean"}
task = "translate" if _translate_to_en else "transcribe"
language = "auto" #@param ['auto', 'af', 'am', 'ar', 'as', 'az', 'ba', 'be', 'bg', 'bn', 'bo', 'br', 'bs', 'ca', 'cs', 'cy', 'da', 'de', 'el', 'en', 'es', 'et', 'eu', 'fa', 'fi', 'fo', 'fr', 'gl', 'gu', 'ha', 'haw', 'he', 'hi', 'hr', 'ht', 'hu', 'hy', 'id', 'is', 'it', 'ja', 'jw', 'ka', 'kk', 'km', 'kn', 'ko', 'la', 'lb', 'ln', 'lo', 'lt', 'lv', 'mg', 'mi', 'mk', 'ml', 'mn', 'mr', 'ms', 'mt', 'my', 'ne', 'nl', 'nn', 'no', 'oc', 'pa', 'pl', 'ps', 'pt', 'ro', 'ru', 'sa', 'sd', 'si', 'sk', 'sl', 'sn', 'so', 'sq', 'sr', 'su', 'sv', 'sw', 'ta', 'te', 'tg', 'th', 'tk', 'tl', 'tr', 'tt', 'uk', 'ur', 'uz', 'vi', 'yi', 'yo', 'yue', 'zh', 'Afrikaans', 'Albanian', 'Amharic', 'Arabic', 'Armenian', 'Assamese', 'Azerbaijani', 'Bashkir', 'Basque', 'Belarusian', 'Bengali', 'Bosnian', 'Breton', 'Bulgarian', 'Burmese', 'Cantonese', 'Castilian', 'Catalan', 'Chinese', 'Croatian', 'Czech', 'Danish', 'Dutch', 'English', 'Estonian', 'Faroese', 'Finnish', 'Flemish', 'French', 'Galician', 'Georgian', 'German', 'Greek', 'Gujarati', 'Haitian', 'Haitian Creole', 'Hausa', 'Hawaiian', 'Hebrew', 'Hindi', 'Hungarian', 'Icelandic', 'Indonesian', 'Italian', 'Japanese', 'Javanese', 'Kannada', 'Kazakh', 'Khmer', 'Korean', 'Lao', 'Latin', 'Latvian', 'Letzeburgesch', 'Lingala', 'Lithuanian', 'Luxembourgish', 'Macedonian', 'Malagasy', 'Malay', 'Malayalam', 'Maltese', 'Maori', 'Marathi', 'Moldavian', 'Moldovan', 'Mongolian', 'Myanmar', 'Nepali', 'Norwegian', 'Nynorsk', 'Occitan', 'Panjabi', 'Pashto', 'Persian', 'Polish', 'Portuguese', 'Punjabi', 'Pushto', 'Romanian', 'Russian', 'Sanskrit', 'Serbian', 'Shona', 'Sindhi', 'Sinhala', 'Sinhalese', 'Slovak', 'Slovenian', 'Somali', 'Spanish', 'Sundanese', 'Swahili', 'Swedish', 'Tagalog', 'Tajik', 'Tamil', 'Tatar', 'Telugu', 'Thai', 'Tibetan', 'Turkish', 'Turkmen', 'Ukrainian', 'Urdu', 'Uzbek', 'Valencian', 'Vietnamese', 'Welsh', 'Yiddish', 'Yoruba']
if language == "auto": language = None


temperature = 0.0 #@param {type:"number", min:0.0}
beam_size = 5 #@param {type:"integer"}
#@markdown - beam_size: temp가 0이 아니면 작동 안함.
best_of = 5 #@param {type:"integer"}
#@markdown - best_of: 후보를 몇개 만들고 선택할지. temp가 0.0이면 거의 의미 없음.
temperature_increment_on_fallback = 0.2 #@param {type:"number"}
compression_ratio_threshold = 2.4 #@param {type:"number"}
#@markdown - compression_ratio_threshold: 반복되는 단어를 억제함. 이 값보다 합축률이 높으면 재시도
_percent_prob_threshold = 10 #@param {type:'number', min:0.01, max:100}
#@markdown - _percent_prob_threshold: 정확도가 이 퍼센트보다 낮으면 실패로 간주(temp 상승 후 재시도). 주어진 값에 로그 취한 후 소숫점 1자리에서 반올림
logprob_threshold = round(math.log(_percent_prob_threshold), 1)
no_speech_threshold = 0.6 #@param {type:"number"}
#@markdown - no_speech_threshold: 로그 정확도 역치가 실패하고, 이 값보다 무음 토큰 가능성이 높을 경우 무음으로 간주.


initial_prompt = "" #@param {type:"string"}
if not initial_prompt: initial_prompt = None
#@markdown - initial_prompt: 초기 프롬프트
_hotwords_split_by_comma = "" #@param {type:"string"}
hotwords = _hotwords_split_by_comma
#@markdown - _hotwords_split_by_comma: `"WhisperX,PyAnnote, GPU"` 같은 식으로 입력 시 해당 단어들의 경향성 증가.
condition_on_previous_text = True #@param {type:"boolean"}

In [ ]:
# @title 결과 파일 설정
output_dir = "/content/output" #@param {type:"string"}
Path(output_dir).mkdir(parents=True, exist_ok=True)
#@markdown - 추후 구글 드라이브 연결을 제작 시 이 위치를 바꿀 수 있음.
#@markdown - 해당 셀 실행시 output_dir의 폴더가 생성됨.
output_format = "srt" #@param ["all","srt","vtt","txt","tsv","json","aud"]
highlight_words = True #@param {type:"boolean"}
#@markdown - highlight_words: True일시 결과물 파일(srt,vtt만 지원)에서 해당 오디오 시점에 밑줄
interpolate_method = "nearest" #@param ["nearest","linear","ignore"]
#@markdown - interpolate_method: 타임스템프를 찾지 못한 단어 처리.
#@markdown - nearest: 가장 가까운 주변 단어의 시간을 가져와 미정렬 단어에 할당
#@markdown - linear: 앞뒤 단어 시간 사이를 선형적으로 나눠 미정렬 단어들에 배분
#@markdown - ignore: 시간 정렬 실패한 단어를 무시하거나 주변 단어에 병합

In [ ]:
# @title 로그 출력 설정
verbose = True #@param {type:"boolean"}
print_progress = True #@param {type:"boolean"}


In [ ]:
# @title VAD 추가설정
vad_method = "pyannote" #@param ["pyannote","silero"]
chunk_size = 30 #@param {type:"integer"}
#@markdown - chunk_size: VAD 조각을 합치는 청크 사이즈. 청크가 너무 길면 줄이기.
vad_onset = 0.5 #@param {type:"number"}
#@markdown - vad_onset: Onset 역치. 발화가 감지되지 않으면 줄이기.
vad_offset = 0.363 #@param {type:"number"}
#@markdown - vad_offset: Offset 역치. 발화가 감지되지 않으면 줄이기.
hf_token = "" #@param {type:"string"}
if not hf_token: hf_token = None
#@markdown - hf_token: Pyannote gated models를 위한 hf 토큰

In [ ]:
# @title 화자 분리 설정
diarize = False #@param {type:"boolean"}
max_speakers = -1 #@param {type:"integer"}
if max_speakers == -1: max_speakers = None
min_speakers = -1 #@param {type:"integer"}
if min_speakers == -1: min_speakers = None
speaker_embeddings = False #@param {type:"boolean"}
#@markdown - speaker_embeddings: json 출력에서 발화자 임베딩 추가


In [ ]:
# @title 결과 문장 분할 설정
no_align = False #@param {type:"boolean"}
return_char_alignments = False #@param {type:"boolean"}
#@markdown - return_char_alignments: 문자 단위 정렬을 json 파일에 반환.
max_line_width = -1 #@param {type:"integer"}
if max_line_width == -1: max_line_width = None
#@markdown - max_line_width: 한 줄의 최대 글자 수(줄을 끊기 전의 최대 글자 수)
max_line_count = -1 #@param {type:"integer"}
if max_line_count == -1: max_line_count = None
#@markdown - max_line_count: segment 내의 최대 줄 수
segment_resolution = "sentence" #@param ["sentence","chunk"]
#@markdown - segment_resoluituon: ~~줄을 끊기 전의 최대 글자 수~~ help에 오타난듯.

In [ ]:
# @title 세부 설정
device_index = 0 #@param {type:"integer"}
patience = 0.1 #@param {type:"number"}
length_penalty = 1.0 #@param {type:"number"}

In [ ]:
# @title 전사 작업 수행

import subprocess
from pathlib import Path
import os

# 값 설정 안할때 비워둬야 하는 옵션

lang_opt = ["--language", str(language)] if language else []
max_speakers_opt = ["--max_speakers", str(max_speakers)] if max_speakers is not None else []
min_speakers_opt = ["--min_speakers", str(min_speakers)] if min_speakers is not None else []
initial_prompt_opt = ["--initial_prompt", str(initial_prompt)] if initial_prompt else []
hotwords_opt = ["--hotwords", str(hotwords)] if hotwords else []
hf_token_opt = ["--hf_token", str(hf_token)] if hf_token else []
max_line_width_opt = ["--max_line_width", str(max_line_width)] if max_line_width is not None else []
max_line_count_opt = ["--max_line_count", str(max_line_count)] if max_line_count is not None else []

# store_true 옵션 

no_align_opt = ["--no_align"] if no_align else []
return_char_alignments_opt = ["--return_char_alignments"] if return_char_alignments else []
diarize_opt = ["--diarize"] if diarize else []
speaker_embeddings_opt = ["--speaker_embeddings"] if speaker_embeddings else []

cmd = [
    "whisperx",
    "--model", str(model),
    "--device", str(device),
    "--batch_size", str(batch_size),
    "--threads", str(threads),
    "--compute_type", str(compute_type),
    "--fp16", str(fp16),
    "--task", str(task),
    "--temperature", str(temperature),
    "--beam_size", str(beam_size),
    "--temperature_increment_on_fallback", str(temperature_increment_on_fallback),
    "--logprob_threshold", str(logprob_threshold),
    "--condition_on_previous_text", str(condition_on_previous_text),
    "--output_dir", str(output_dir),
    "--output_format", str(output_format),
    "--interpolate_method", str(interpolate_method),
    "--highlight_words", str(highlight_words),
    "--verbose", str(verbose),
    "--print_progress", str(print_progress),
    "--vad_method", str(vad_method),
    "--chunk_size", str(chunk_size),
    "--vad_onset", str(vad_onset),
    "--vad_offset", str(vad_offset),
    "--segment_resolution", str(segment_resolution),
    "--device_index", str(device_index),
    "--best_of", str(best_of),
    "--patience", str(patience),
    "--length_penalty", str(length_penalty),
]

cmd += lang_opt
cmd += initial_prompt_opt
cmd += hotwords_opt
cmd += hf_token_opt
cmd += max_speakers_opt
cmd += min_speakers_opt
cmd += max_line_width_opt
cmd += max_line_count_opt

cmd += diarize_opt
cmd += speaker_embeddings_opt
cmd += no_align_opt
cmd += return_char_alignments_opt

cmd += inputs

print(f'실행 명령어: {" ".join(cmd)}')

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = None

try:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
        env=env
    )

    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

except KeyboardInterrupt:
    print("\n사용자 중단")
    if process and process.poll() is None:
        process.terminate()
    raise

finally:
    if process and process.poll() is None:
        process.kill()

In [ ]:
# @title 파일 저장
from google.colab import files
mode = "download each file" #@param ["download each file", "folder zip"]
zip_name = "output.zip" #@param {type:"string"}
if mode == "download each file":
    lst = [str(f) for f in Path(output_dir).rglob("*") if f.is_file()] # 재귀적으로 순회할 필요 자체는 없긴 한데. yt-dlp와 다르게 여긴 여러개가 불가능.
    total = len(lst)
    for idx, f in enumerate(lst):
        print(f"다운로드 {idx + 1}/{total}개: {f}")
        files.download(f)
elif mode == "folder zip":
    !zip -r {zip_name} {output_dir}
    files.download(zip_name)
